# Generating a Subset of Haunted Places Data for Radial Chart Visualization:
## Religion and Apparition Type Analysis

This script processes a subset of the haunted places dataset to create a cleaned JSON file (radial_data.json) for use in a D3 radial stacked bar chart. It extracts and summarizes key columns: Religion_Intersection, Distance_to_Nearest_Worship, and Apparition_Type.

Because the original apparition types were large in number and often repetitive, the code groups them into broader, more meaningful categories like "Ghost," "Spirit," "Demon," "Orb," "Entity/Light," and "Unknown" using text matching. This grouping makes the data easier to represent visually and track across religions.

The script then groups the data by religion, calculates each religion’s median distance to the nearest place of worship, and computes the percentage breakdown of apparition types within each group.

The resulting JSON powers an interactive D3 radial stacked bar chart (Which Religions Are Closest to Haunted Places, and What Apparitions Appear Near Them?), allowing users to explore how different spiritual traditions shape haunting reports through interactive tooltips, zooming, and apparition breakdowns.

In [13]:
import pandas as pd
import json
import re
from pathlib import Path

# Load dataset
df = pd.read_csv("../data/processed/haunted_places_features_added_v2.tab", sep="\t")
print(f"Original rows: {len(df)}")

# Select and clean
df = df[["Religion_Intersection", "Distance_to_Nearest_Worship", "Apparition_Type"]]
df = df.dropna()
print(f"Rows after dropna: {len(df)}")

# Apparition type grouping logic
primary_types = {
    "demon": "Demon",
    "ghost": "Ghost",
    "spirit": "Spirit",
    "phantom": "Phantom",
    "orb": "Orb",
    "floating light": "Entity/Light",
    "evil presence": "Entity/Light",
    "dark entity": "Entity/Light",
    "ufo": "Unknown",
    "unknown": "Unknown"
}

def classify_apparition(text):
    found = set()
    text = str(text).lower()
    for key, label in primary_types.items():
        if re.search(rf"\b{re.escape(key)}\b", text):
            found.add(label)
    if not found:
        return "Other"
    elif len(found) == 1:
        return found.pop()
    else:
        return "Multiple Apparitions"

df["Grouped_Apparition"] = df["Apparition_Type"].apply(classify_apparition)
print(f"Unique apparition types: {df['Grouped_Apparition'].nunique()}")

# Group by religion
religion_groups = df.groupby("Religion_Intersection")
print(f"Number of religions found: {len(religion_groups)}")

# Generate JSON structure
radial_data = []

for religion, group in religion_groups:
    median_distance = group["Distance_to_Nearest_Worship"].median()
    apparition_counts = group["Grouped_Apparition"].value_counts(normalize=True) * 100
    apparition_dict = apparition_counts.round(1).to_dict()

    radial_data.append({
        "religion": religion,
        "median_distance": round(median_distance, 2),
        "apparitions": apparition_dict
    })

print(f"Radial data entries created: {len(radial_data)}")

# Output directory
output_dir = Path("../viz/radial_chart")
output_dir.mkdir(parents=True, exist_ok=True)

# Save JSON
output_file = output_dir / "radial_data.json"
with open(output_file, "w") as f:
    json.dump(radial_data, f, indent=2)

print(f"Saved radial_data.json to {output_file.resolve()}")

Original rows: 10992
Rows after dropna: 10417
Unique apparition types: 9
Number of religions found: 22
Radial data entries created: 22
Saved radial_data.json to /Users/serafinasmith/dsci_550_a1/viz/radial_chart/radial_data.json
